# OFL Competency Question Evaluation

This notebook evaluates the **Ontology of Flaps in Plastic and Reconstructive Surgery (OFL v2.1.0)** against its competency questions (CQ1–CQ17) and five task-based use cases via ABox queries.

| File | Role |
|------|------|
| `ofl_2.1.0_inferred.owl` | Inferred TBox (OWL/RDF) |
| `ofl_2.1.0_abox_inferred.owl` | Fully reasoned ABox (OWL RL) |
| `fma_obo.owl` | FMA — label resolution only |

**Reproduce:** install `rdflib` and `pandas`, place the three files in `ontologies/`, then run all cells from top to bottom.

In [ ]:
from rdflib import Graph, Namespace, URIRef, RDF, RDFS, OWL
from rdflib.namespace import OWL as OWL_NS
import pandas as pd
from IPython.display import display
from pathlib import Path

OFL = Namespace('https://purl.bioontology.org/ontology/OFL/')
OBO = Namespace('http://purl.obolibrary.org/obo/')

PREFIXES = """
PREFIX rdf:     <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs:    <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:     <http://www.w3.org/2002/07/owl#>
PREFIX obo:     <http://purl.obolibrary.org/obo/>
PREFIX ofl:     <https://purl.bioontology.org/ontology/OFL/>
PREFIX dcterms: <http://purl.org/dc/terms/>
"""

ONT           = Path('..') / 'ontologies'
BFO           = ONT / 'bfo-core.owl'
OBI           = ONT / 'obi.owl'
RO            = ONT / 'ro.owl'
COB           = ONT / 'cob.owl'
TBOX          = ONT / 'ofl_2.1.0_inferred.owl'
ABOX_INFERRED = ONT / 'ofl_2.1.0_abox_inferred.owl'
FMA           = ONT / 'fma_obo.owl' 

g = Graph()
g.parse(str(TBOX), format='xml')
g.parse(str(BFO), format='xml')
g.parse(str(OBI), format='xml')
g.parse(str(RO), format='xml')
g.parse(str(COB), format='xml')
g.parse(str(ABOX_INFERRED), format='xml')
g.parse(str(FMA), format='xml')
print(f'Total: {len(g):,} triples')

Total: 6,284,450 triples


In [66]:
def run(sparql, limit=20, title="", graph=None):
    if graph is None:
        graph = g
    result = graph.query(PREFIXES + sparql)
    cols = [str(v) for v in result.vars]
    rows = list(result)
    data = [{c: (str(getattr(r, c)) if getattr(r, c) is not None else "—") for c in cols} for r in rows]
    df = pd.DataFrame(data, columns=cols)
    if title:
        print(f"\n{'='*60}")
        print(title)
    print(f"  {len(rows):,} row(s)" + (f" (showing first {limit})" if len(rows) > limit else ""))
    display(df.head(limit))
    return df

In [67]:
from rdflib import RDFS, URIRef

# Only types actually asserted in the graph — keeps VALUES strings small
_all_types = set(g.objects(None, RDF.type))

def _cls_vals(*roots):
    """Transitive subclasses of roots that appear as types in the graph."""
    subs = set(roots)
    for r in roots:
        subs.update(c for c in g.transitive_subjects(RDFS.subClassOf, r) if isinstance(c, URIRef))
    used = subs & _all_types or subs
    return " ".join(f"<{c}>" for c in used)

def _pair_vals(*roots):
    """(root, subclass) pairs where subclass appears as a type in the graph."""
    pairs, seen = [], set()
    for r in roots:
        subs = {r} | {c for c in g.transitive_subjects(RDFS.subClassOf, r) if isinstance(c, URIRef)}
        for sub in subs & _all_types:
            if (r, sub) not in seen:
                seen.add((r, sub))
                pairs.append(f"(<{r}> <{sub}>)")
    return " ".join(pairs) or " ".join(f"(<{r}> <{r}>)" for r in roots)

flap_cls       = _cls_vals(OFL.OFLID10002)
pedicle_cls    = _cls_vals(OFL.OFLID106008)
mn_cls         = _cls_vals(OFL.OFLID10135)
distance_cls   = _cls_vals(OFL.OFLID10086)
chimeric_cls   = _cls_vals(OFL.OFLID1000093)
island_cls     = _cls_vals(OFL.OFLID10115)
preharvest_cls      = _cls_vals(OFL.OFLID10045)
insertion_prep_cls  = _cls_vals(OFL.OFLID130001)
skingraft_cls       = _cls_vals(OFL.OFLID120066)
anast_cls      = _cls_vals(OFL.OFLID1000170, OFL.OFLID1000175)
pedicled_cls   = _cls_vals(OFL.OFLID1000138)
fma_vessel_cls = _cls_vals(OBO.FMA_86187, OBO.FMA_86188)

nakajima_pairs = _pair_vals(OFL.OFLID10075, OFL.OFLID10076, OFL.OFLID10078)
transfer_pairs = _pair_vals(OFL.OFLID1000137, OFL.OFLID1000138)
vasc_pairs     = _pair_vals(OFL.OFLID10004, OFL.OFLID10077, OFL.OFLID10076,
                             OFL.OFLID10075, OFL.OFLID10078)
flow_pairs     = _pair_vals(OFL.OFLID10113, OFL.OFLID10123)
movement_pairs = _pair_vals(OFL.OFLID10014, OFL.OFLID10067, OFL.OFLID10070, OFL.OFLID10132)

_mn_direct     = [c for c in g.subjects(RDFS.subClassOf, OFL.OFLID10135) if isinstance(c, URIRef)]
mn_task3_pairs = _pair_vals(*_mn_direct)

# backward-compat alias used by CQ1
flap_cls_values = flap_cls

print("Class sets precomputed.")

Class sets precomputed.


---
## Competency Questions

### CQ1 — Anatomical Composition
*What anatomical structures compose the flap?*

In [22]:
_cq1 = """
SELECT ?flaplabel ?componentLabel ?typeLabel
WHERE {
  VALUES ?cls { FLAP_CLS_PLACEHOLDER }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        ofl:OFLID13296 ?component .
  OPTIONAL { ?flap      rdfs:label ?flaplabel }
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
} LIMIT 10
"""
run(_cq1.replace("FLAP_CLS_PLACEHOLDER", flap_cls_values),
    title="CQ1 ABox — anatomical components of generated flaps")


CQ1 ABox — anatomical components of generated flaps
  10 row(s)


,flaplabel,componentLabel,typeLabel
0,Right Serratus anterior muscle flap with subcu...,Flap tissue of Right Serratus anterior muscle ...,Anatomical structure
1,Right Serratus anterior muscle flap with subcu...,Right Subcutaneous adipose tissue of back of t...,Subcutaneous adipose tissue
2,Right Latissimus dorsi flap with serratus ante...,Flap tissue of Right Latissimus dorsi flap wit...,Anatomical structure
3,Right Latissimus dorsi flap with serratus ante...,Right structure 13397 of patient 530,Serratus anterior
4,Left Serratus anterior muscle flap with subcut...,Flap tissue of Left Serratus anterior muscle f...,Anatomical structure
5,Left Serratus anterior muscle flap with subcut...,Left Subcutaneous adipose tissue of back of th...,Subcutaneous adipose tissue
6,Right Serratus anterior muscle flap with subcu...,Flap tissue of Right Serratus anterior muscle ...,Anatomical structure
7,Right Serratus anterior muscle flap with subcu...,Right Subcutaneous adipose tissue of back of t...,Subcutaneous adipose tissue
8,Left Serratus anterior fascia flap with superf...,Flap tissue of Left Serratus anterior fascia f...,Anatomical structure
9,Left Serratus anterior fascia flap with superf...,Left structure 32583 of patient 624,Superficial fascia of back


,flaplabel,componentLabel,typeLabel
0,Right Serratus anterior muscle flap with subcu...,Flap tissue of Right Serratus anterior muscle ...,Anatomical structure
1,Right Serratus anterior muscle flap with subcu...,Right Subcutaneous adipose tissue of back of t...,Subcutaneous adipose tissue
2,Right Latissimus dorsi flap with serratus ante...,Flap tissue of Right Latissimus dorsi flap wit...,Anatomical structure
3,Right Latissimus dorsi flap with serratus ante...,Right structure 13397 of patient 530,Serratus anterior
4,Left Serratus anterior muscle flap with subcut...,Flap tissue of Left Serratus anterior muscle f...,Anatomical structure
5,Left Serratus anterior muscle flap with subcut...,Left Subcutaneous adipose tissue of back of th...,Subcutaneous adipose tissue
6,Right Serratus anterior muscle flap with subcu...,Flap tissue of Right Serratus anterior muscle ...,Anatomical structure
7,Right Serratus anterior muscle flap with subcu...,Right Subcutaneous adipose tissue of back of t...,Subcutaneous adipose tissue
8,Left Serratus anterior fascia flap with superf...,Flap tissue of Left Serratus anterior fascia f...,Anatomical structure
9,Left Serratus anterior fascia flap with superf...,Left structure 32583 of patient 624,Superficial fascia of back


### CQ2 — Size
*What is the size of the flap?*

In [23]:
_cq2 = """
SELECT ?flapLabel ?qualityLabel
WHERE {
  VALUES ?cls { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        obo:RO_0000086 ?quality .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?quality rdfs:label ?qualityLabel }
} LIMIT 10
"""
run(_cq2.replace("CLS", flap_cls),
    title="CQ2 ABox — flap individuals with size qualities (volume / mass)")


CQ2 ABox — flap individuals with size qualities (volume / mass)
  10 row(s)


,flapLabel,qualityLabel
0,Right Serratus anterior muscle flap with subcu...,Volume of Right Serratus anterior muscle flap ...
1,Right Serratus anterior muscle flap with subcu...,Mass of Right Serratus anterior muscle flap wi...
2,Right Latissimus dorsi flap with serratus ante...,Volume of Right Latissimus dorsi flap with ser...
3,Right Latissimus dorsi flap with serratus ante...,Mass of Right Latissimus dorsi flap with serra...
4,Left Serratus anterior muscle flap with subcut...,Volume of Left Serratus anterior muscle flap w...
5,Left Serratus anterior muscle flap with subcut...,Mass of Left Serratus anterior muscle flap wit...
6,Right Serratus anterior muscle flap with subcu...,Volume of Right Serratus anterior muscle flap ...
7,Right Serratus anterior muscle flap with subcu...,Mass of Right Serratus anterior muscle flap wi...
8,Left Serratus anterior fascia flap with superf...,Volume of Left Serratus anterior fascia flap w...
9,Left Serratus anterior fascia flap with superf...,Mass of Left Serratus anterior fascia flap wit...


,flapLabel,qualityLabel
0,Right Serratus anterior muscle flap with subcu...,Volume of Right Serratus anterior muscle flap ...
1,Right Serratus anterior muscle flap with subcu...,Mass of Right Serratus anterior muscle flap wi...
2,Right Latissimus dorsi flap with serratus ante...,Volume of Right Latissimus dorsi flap with ser...
3,Right Latissimus dorsi flap with serratus ante...,Mass of Right Latissimus dorsi flap with serra...
4,Left Serratus anterior muscle flap with subcut...,Volume of Left Serratus anterior muscle flap w...
5,Left Serratus anterior muscle flap with subcut...,Mass of Left Serratus anterior muscle flap wit...
6,Right Serratus anterior muscle flap with subcu...,Volume of Right Serratus anterior muscle flap ...
7,Right Serratus anterior muscle flap with subcu...,Mass of Right Serratus anterior muscle flap wi...
8,Left Serratus anterior fascia flap with superf...,Volume of Left Serratus anterior fascia flap w...
9,Left Serratus anterior fascia flap with superf...,Mass of Left Serratus anterior fascia flap wit...


### CQ3 — Vessel Connection
*To which vessels is the flap connected?*

In [71]:
_cq3 = """
SELECT ?flapLabel ?pedicleLabel ?vesselLabel
WHERE {
  VALUES ?cls { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        ofl:OFLID13296 ?pedicle .
  ?pedicle ofl:OFLID13296 ?vessel .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?pedicle rdfs:label ?pedicleLabel }
  OPTIONAL { ?vessel  rdfs:label ?vesselLabel }
} LIMIT 10
"""
run(_cq3.replace("CLS", _cls_vals(OFL.OFLID10002)),
    title="CQ3 ABox — flap individuals with their pedicle part and vessels")


CQ3 ABox — flap individuals with their pedicle part and vessels
  10 row(s)


,flapLabel,pedicleLabel,vesselLabel
0,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18871 of patient 301
1,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18911 of patient 301
2,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle artery of Left Inferior gluteal artery...
3,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle vein of Left Inferior gluteal artery p...
4,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Arterial perforator of a vessel in patient htt...
5,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Venous perforator of Left Inferior gluteal art...
6,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18871 of patient 301
7,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18911 of patient 301
8,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle artery of Left Inferior gluteal artery...
9,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle vein of Left Inferior gluteal artery p...


,flapLabel,pedicleLabel,vesselLabel
0,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18871 of patient 301
1,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18911 of patient 301
2,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle artery of Left Inferior gluteal artery...
3,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle vein of Left Inferior gluteal artery p...
4,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Arterial perforator of a vessel in patient htt...
5,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Venous perforator of Left Inferior gluteal art...
6,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18871 of patient 301
7,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Left vessel 18911 of patient 301
8,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle artery of Left Inferior gluteal artery...
9,Left Inferior gluteal artery perforator flap o...,Flap pedicle of Left Inferior gluteal artery p...,Pedicle vein of Left Inferior gluteal artery p...


### CQ4 — Mathes and Nahai Classification
*What is the Mathes and Nahai classification of the muscle flap?*

In [47]:
_cq4a = """
SELECT ?flapLabel ?originLabel
WHERE {
  VALUES ?cls { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?cls ;
        ofl:OFLID12003 ?origin .
  OPTIONAL { ?flap   rdfs:label ?flapLabel }
  OPTIONAL { ?origin rdfs:label ?originLabel }
} LIMIT 10
"""
run(_cq4a.replace("CLS", _cls_vals(OFL.OFLID10135)),
    title="CQ4 ABox — individuals classified as Mathes-Nahai Type V")


CQ4 ABox — individuals classified as Mathes-Nahai Type V
  10 row(s)


,flapLabel,originLabel
0,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
1,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
2,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
3,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
4,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
5,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
6,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
7,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
8,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
9,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...


,flapLabel,originLabel
0,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
1,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
2,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
3,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
4,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
5,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
6,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
7,Right Rectus abdominis flap with seventh rib o...,Donor site of Right Rectus abdominis flap with...
8,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...
9,Left Rectus abdominis flap with seventh rib of...,Donor site of Left Rectus abdominis flap with ...


In [26]:
_cq4b = """
SELECT ?flapLabel ?typeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?typeLabel }
} LIMIT 10
"""
run(_cq4b.replace("CLS", mn_cls),
    title="CQ4 ABox — flap individuals by Mathes-Nahai type")


CQ4 ABox — flap individuals by Mathes-Nahai type
  10 row(s)


,flapLabel,typeLabel
0,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
1,Left Rectus abdominis flap with seventh rib of...,Rectus abdominis flap with seventh rib
2,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
3,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
4,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
5,Left Rectus abdominis flap with seventh rib of...,Rectus abdominis flap with seventh rib
6,Left Rectus abdominis flap with seventh rib of...,Rectus abdominis flap with seventh rib
7,Component sub-flap of Chimeric Rectus abdomini...,Rectus abdominis flap with seventh rib
8,Chimeric Rectus abdominis flap with seventh ri...,Rectus abdominis flap with seventh rib
9,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib


,flapLabel,typeLabel
0,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
1,Left Rectus abdominis flap with seventh rib of...,Rectus abdominis flap with seventh rib
2,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
3,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
4,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib
5,Left Rectus abdominis flap with seventh rib of...,Rectus abdominis flap with seventh rib
6,Left Rectus abdominis flap with seventh rib of...,Rectus abdominis flap with seventh rib
7,Component sub-flap of Chimeric Rectus abdomini...,Rectus abdominis flap with seventh rib
8,Chimeric Rectus abdominis flap with seventh ri...,Rectus abdominis flap with seventh rib
9,Right Rectus abdominis flap with seventh rib o...,Rectus abdominis flap with seventh rib


### CQ5 — Nakajima Classification
*What is the Nakajima²⁴ classification of the flap?*

In [45]:
_cq5 = """
SELECT ?flapLabel ?nakajimaLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?nakajimaLabel }
} LIMIT 10
"""
run(_cq5.replace("CLS", _cls_vals(OFL.OFLID10078)),
    title="CQ5 ABox — musculocutaneous perforator-based flap individuals (Nakajima)")


CQ5 ABox — musculocutaneous perforator-based flap individuals (Nakajima)
  10 row(s)


,flapLabel,nakajimaLabel
0,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
1,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
2,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
3,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
4,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
5,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
6,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
7,Chimeric Deep inferior epigastric perforator f...,Deep inferior epigastric perforator flap with ...
8,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
9,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...


,flapLabel,nakajimaLabel
0,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
1,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
2,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
3,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
4,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
5,Right Deep inferior epigastric perforator flap...,Deep inferior epigastric perforator flap with ...
6,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
7,Chimeric Deep inferior epigastric perforator f...,Deep inferior epigastric perforator flap with ...
8,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...
9,Left Deep inferior epigastric perforator flap ...,Deep inferior epigastric perforator flap with ...


### CQ6 — Transfer Distance
*Is the flap local, regional or distant?*

In [28]:
_cq6 = """
SELECT ?flapLabel ?distLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?distLabel }
} LIMIT 10
"""
run(_cq6.replace("CLS", _cls_vals(OFL.OFLID10134)),
    title="CQ6 ABox — local flap individuals (OFLID10134)")


CQ6 ABox — local flap individuals (OFLID10134)
  6 row(s)


,flapLabel,distLabel
0,Left Local flaps of patient 1995,Local flaps
1,Left Local flaps of patient 1996,Local flaps
2,Right Local flaps of patient 1999,Local flaps
3,Right Local flaps of patient 1998,Local flaps
4,Left Local flaps of patient 1997,Local flaps
5,Right Local flaps of patient 2000,Local flaps


,flapLabel,distLabel
0,Left Local flaps of patient 1995,Local flaps
1,Left Local flaps of patient 1996,Local flaps
2,Right Local flaps of patient 1999,Local flaps
3,Right Local flaps of patient 1998,Local flaps
4,Left Local flaps of patient 1997,Local flaps
5,Right Local flaps of patient 2000,Local flaps


### CQ7 — Transfer Method
*Is the flap pedicled or free?*

In [46]:
_cq7 = """
SELECT ?flapLabel ?typeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?typeLabel }
} LIMIT 10
"""
run(_cq7.replace("CLS", _cls_vals(OFL.OFLID10191)),
    title="CQ7 ABox — free flap individuals (OFLID10054)")


CQ7 ABox — free flap individuals (OFLID10054)
  10 row(s)


,flapLabel,typeLabel
0,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
1,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
2,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
3,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
4,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
5,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
6,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
7,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
8,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
9,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum


,flapLabel,typeLabel
0,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
1,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
2,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
3,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
4,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
5,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
6,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
7,Right Pectoralis major flap with sternum of pa...,Pectoralis major flap with sternum
8,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum
9,Left Pectoralis major flap with sternum of pat...,Pectoralis major flap with sternum


### CQ8 — Vascular Pattern
*Is it a random pattern or an axial or perforator flap?*

In [30]:
_cq8 = """
SELECT ?flapLabel ?bloodSupplyLabel
WHERE {
  VALUES (?bloodSupplyCls ?t) { PAIRS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap           rdfs:label ?flapLabel }
  OPTIONAL { ?bloodSupplyCls rdfs:label ?bloodSupplyLabel }
} LIMIT 10
"""
run(_cq8.replace("PAIRS", vasc_pairs),
    title="CQ8 ABox — flap individuals by vascular pattern")


CQ8 ABox — flap individuals by vascular pattern
  10 row(s)


,flapLabel,bloodSupplyLabel
0,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
1,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
2,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
3,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
4,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
5,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
6,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
7,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
8,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
9,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps


,flapLabel,bloodSupplyLabel
0,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
1,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
2,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
3,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
4,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
5,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
6,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
7,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps
8,Right Vy flaps of the fingertip flap with subc...,Random pattern flaps
9,Left Vy flaps of the fingertip flap with subcu...,Random pattern flaps


### CQ9 — Chimeric Flap
*Is it a chimeric flap?*

In [31]:
_cq9 = """
SELECT ?flapLabel ?chimericTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?chimericTypeLabel }
} LIMIT 10
"""
run(_cq9.replace("CLS", chimeric_cls),
    title="CQ9 ABox — flap individuals by chimeric type")


CQ9 ABox — flap individuals by chimeric type
  10 row(s)


,flapLabel,chimericTypeLabel
0,Chimeric Deep inferior epigastric perforator f...,Type i: classical chimerism
1,Chimeric Rectus abdominis flap with seventh ri...,Type i: classical chimerism
2,Chimeric Jejunum flap with myenteric nerve ple...,Type i: classical chimerism
3,Chimeric Omental flap with lymphoid tissue of ...,Type i: classical chimerism
4,Chimeric Omental flap with omentum of patient ...,Type iii: perforator chimerism
5,Chimeric Rectus abdominis flap with ninth rib ...,Type iii: perforator chimerism
6,Chimeric Rectus abdominis flap with skin of ab...,Type iii: perforator chimerism
7,Chimeric Deep inferior epigastric perforator f...,Type iii: perforator chimerism
8,Chimeric Deep inferior epigastric perforator f...,Flaps classified by chimeric type
9,Chimeric Deep inferior epigastric perforator f...,Flaps classified by chimeric type


,flapLabel,chimericTypeLabel
0,Chimeric Deep inferior epigastric perforator f...,Type i: classical chimerism
1,Chimeric Rectus abdominis flap with seventh ri...,Type i: classical chimerism
2,Chimeric Jejunum flap with myenteric nerve ple...,Type i: classical chimerism
3,Chimeric Omental flap with lymphoid tissue of ...,Type i: classical chimerism
4,Chimeric Omental flap with omentum of patient ...,Type iii: perforator chimerism
5,Chimeric Rectus abdominis flap with ninth rib ...,Type iii: perforator chimerism
6,Chimeric Rectus abdominis flap with skin of ab...,Type iii: perforator chimerism
7,Chimeric Deep inferior epigastric perforator f...,Type iii: perforator chimerism
8,Chimeric Deep inferior epigastric perforator f...,Flaps classified by chimeric type
9,Chimeric Deep inferior epigastric perforator f...,Flaps classified by chimeric type


### CQ10 — Arterial Flow Direction
*What is the arterial flow direction (e.g. reversed flow or flow-through flap)?*

In [53]:
_cq10 = """
SELECT DISTINCT ?flap ?flapLabel ?type ?typeLabel ?rootLabel
WHERE {
  VALUES ?root { CLS }

  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?type .

  ?type rdfs:subClassOf* ?root .

  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?type rdfs:label ?typeLabel }
  OPTIONAL { ?root rdfs:label ?rootLabel }
}
LIMIT 10
"""

run(
    _cq10.replace("CLS", _cls_vals(OFL.OFLID10093)),
    title="CQ10 ABox — individuals under OFLID10093"
)


CQ10 ABox — individuals under OFLID10093
  0 row(s)


,flap,flapLabel,type,typeLabel,rootLabel


,flap,flapLabel,type,typeLabel,rootLabel


### CQ11 — Movement Type
*Is it a rotational, advancement or transpositional flap?*

In [33]:
_cq11 = """
SELECT ?flapLabel ?movementLabel
WHERE {
  VALUES (?movement ?t) { PAIRS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap     rdfs:label ?flapLabel }
  OPTIONAL { ?movement rdfs:label ?movementLabel }
} LIMIT 10
"""
run(_cq11.replace("PAIRS", movement_pairs),
    title="CQ11 ABox — flap individuals by movement type")


CQ11 ABox — flap individuals by movement type
  10 row(s)


,flapLabel,movementLabel
0,Right Gracilis flap with skin of medial part o...,Rotation flaps
1,Right Omental flap with omentum of patient 68,Rotation flaps
2,Left Deltopectoral flap with fascia of deltoid...,Rotation flaps
3,Right Groin flap with abdominal fascia of pati...,Rotation flaps
4,Left Pectoralis major flap with sternum of pat...,Rotation flaps
5,Left Tensor fasciae latae flap with subcutaneo...,Rotation flaps
6,Left Iliac crest flap with skin of hip of pati...,Rotation flaps
7,Left Lateral arm flap with brachialis of patie...,Rotation flaps
8,Right Rectus abdominis flap with eighth rib of...,Rotation flaps
9,Right Deltopectoral flap with pectoralis major...,Rotation flaps


,flapLabel,movementLabel
0,Right Gracilis flap with skin of medial part o...,Rotation flaps
1,Right Omental flap with omentum of patient 68,Rotation flaps
2,Left Deltopectoral flap with fascia of deltoid...,Rotation flaps
3,Right Groin flap with abdominal fascia of pati...,Rotation flaps
4,Left Pectoralis major flap with sternum of pat...,Rotation flaps
5,Left Tensor fasciae latae flap with subcutaneo...,Rotation flaps
6,Left Iliac crest flap with skin of hip of pati...,Rotation flaps
7,Left Lateral arm flap with brachialis of patie...,Rotation flaps
8,Right Rectus abdominis flap with eighth rib of...,Rotation flaps
9,Right Deltopectoral flap with pectoralis major...,Rotation flaps


### CQ12 — Island Flap
*Is it an island flap?*

In [72]:
#island flap class: OFLID120032
#Surgical flap class: OFLID10002

_cq12 = """
SELECT DISTINCT ?flap ?flapLabel ?islandLabel
WHERE {
  VALUES ?t { CLS }

  ?flap rdf:type ?t .

  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t rdfs:label ?islandLabel }
}
LIMIT 10
"""

run(
    _cq12.replace("CLS", _cls_vals(OFL.OFLID120032)),
    title="CQ12 ABox — direct island flap individuals"
)



CQ12 ABox — direct island flap individuals
  4 row(s)


,flap,flapLabel,islandLabel
0,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the right pectoral re...,Cutaneous island flap
1,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the left pectoral reg...,Cutaneous island flap
2,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the left pectoral reg...,Cutaneous island flap
3,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the right pectoral re...,Cutaneous island flap


,flap,flapLabel,islandLabel
0,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the right pectoral re...,Cutaneous island flap
1,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the left pectoral reg...,Cutaneous island flap
2,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the left pectoral reg...,Cutaneous island flap
3,https://purl.bioontology.org/ontology/OFL/OFLI...,Cutaneous island flap of the right pectoral re...,Cutaneous island flap


### CQ13 — Insertion Site Preparation
*How was the insertion site prepared?*

In [35]:
_cq13 = """
SELECT ?flapLabel ?prepTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?prepTypeLabel }
} LIMIT 10
"""
run(_cq13.replace("CLS", insertion_prep_cls),
    title="CQ13 ABox — flap individuals by insertion site preparation type")


CQ13 ABox — flap individuals by insertion site preparation type
  0 row(s)


,flapLabel,prepTypeLabel


,flapLabel,prepTypeLabel


### CQ14 — Pre-Harvest Modification
*Was the flap modified before harvesting?*

In [36]:
_cq14 = """
SELECT ?flapLabel ?modLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?modLabel }
} LIMIT 10
"""
run(_cq14.replace("CLS", preharvest_cls),
    title="CQ14 ABox — pre-harvest modified flap individuals")


CQ14 ABox — pre-harvest modified flap individuals
  10 row(s)


,flapLabel,modLabel
0,Right Omental flap with omentum of patient 68,Flaps without preharvest modification
1,Right Scapular flap with thoracic fascia of pa...,Flaps without preharvest modification
2,Left Gluteus flap with gluteus maximus of pati...,Flaps without preharvest modification
3,Right Omohyoid flap with ansa cervicalis of pa...,Flaps without preharvest modification
4,Right Scapular flap with skin of back of patie...,Flaps without preharvest modification
5,Right Groin flap with abdominal fascia of pati...,Flaps without preharvest modification
6,Left Lateral arm flap with lateral head of tri...,Flaps without preharvest modification
7,Left Deltopectoral flap with fascia of deltoid...,Flaps without preharvest modification
8,Right Groin flap with abdominal fascia of pati...,Flaps without preharvest modification
9,Left Tensor fasciae latae flap with subcutaneo...,Flaps without preharvest modification


,flapLabel,modLabel
0,Right Omental flap with omentum of patient 68,Flaps without preharvest modification
1,Right Scapular flap with thoracic fascia of pa...,Flaps without preharvest modification
2,Left Gluteus flap with gluteus maximus of pati...,Flaps without preharvest modification
3,Right Omohyoid flap with ansa cervicalis of pa...,Flaps without preharvest modification
4,Right Scapular flap with skin of back of patie...,Flaps without preharvest modification
5,Right Groin flap with abdominal fascia of pati...,Flaps without preharvest modification
6,Left Lateral arm flap with lateral head of tri...,Flaps without preharvest modification
7,Left Deltopectoral flap with fascia of deltoid...,Flaps without preharvest modification
8,Right Groin flap with abdominal fascia of pati...,Flaps without preharvest modification
9,Left Tensor fasciae latae flap with subcutaneo...,Flaps without preharvest modification


### CQ15 — Split-Thickness Skin Grafting
*Did the flap require split-thickness skin grafting?*

In [37]:
_cq15 = """
SELECT ?flapLabel ?graftTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?t .
  OPTIONAL { ?flap rdfs:label ?flapLabel }
  OPTIONAL { ?t    rdfs:label ?graftTypeLabel }
} LIMIT 10
"""
run(_cq15.replace("CLS", skingraft_cls),
    title="CQ15 ABox — flap individuals requiring split-thickness skin grafting")


CQ15 ABox — flap individuals requiring split-thickness skin grafting
  10 row(s)


,flapLabel,graftTypeLabel
0,Left Gracilis flap with fascia of thigh of pat...,Flaps with skin graft
1,Right Scapular flap with thoracic fascia of pa...,Flaps with skin graft
2,Left Gluteus flap with gluteus maximus of pati...,Flaps with skin graft
3,Right Omohyoid flap with ansa cervicalis of pa...,Flaps with skin graft
4,Right Gracilis flap with fascia of thigh of pa...,Flaps with skin graft
5,Right Groin flap with abdominal fascia of pati...,Flaps with skin graft
6,Left Pectoralis major flap with sternum of pat...,Flaps with skin graft
7,Left Tensor fasciae latae flap with subcutaneo...,Flaps with skin graft
8,Left Gluteal thigh flap of patient 363,Flaps with skin graft
9,Right Gluteal artery perforator flap with subc...,Flaps with skin graft


,flapLabel,graftTypeLabel
0,Left Gracilis flap with fascia of thigh of pat...,Flaps with skin graft
1,Right Scapular flap with thoracic fascia of pa...,Flaps with skin graft
2,Left Gluteus flap with gluteus maximus of pati...,Flaps with skin graft
3,Right Omohyoid flap with ansa cervicalis of pa...,Flaps with skin graft
4,Right Gracilis flap with fascia of thigh of pa...,Flaps with skin graft
5,Right Groin flap with abdominal fascia of pati...,Flaps with skin graft
6,Left Pectoralis major flap with sternum of pat...,Flaps with skin graft
7,Left Tensor fasciae latae flap with subcutaneo...,Flaps with skin graft
8,Left Gluteal thigh flap of patient 363,Flaps with skin graft
9,Right Gluteal artery perforator flap with subc...,Flaps with skin graft


### CQ16 — Vessel Anastomosis
*How was vessel anastomosis performed?*

In [38]:
_cq16 = """
SELECT ?anastLabel ?anastomosisTypeLabel
WHERE {
  VALUES ?t { CLS }
  ?anast rdf:type owl:NamedIndividual ;
         rdf:type ?t .
  OPTIONAL { ?anast rdfs:label ?anastLabel }
  OPTIONAL { ?t     rdfs:label ?anastomosisTypeLabel }
} LIMIT 10
"""
run(_cq16.replace("CLS", anast_cls),
    title="CQ16 ABox — vessel anastomosis individuals by type")


CQ16 ABox — vessel anastomosis individuals by type
  10 row(s)


,anastLabel,anastomosisTypeLabel
0,Venous anastomosis of Right Serratus anterior ...,Venous anastomosis
1,Venous anastomosis of Left Thoracodorsal arter...,Venous anastomosis
2,Venous anastomosis of Left Posterior tibial ar...,Venous anastomosis
3,Venous anastomosis of Right Lateral thoracic p...,Venous anastomosis
4,Venous anastomosis of Right Serratus anterior ...,Venous anastomosis
5,Venous anastomosis of Left Gluteal artery perf...,Venous anastomosis
6,Venous anastomosis of Left Thoracodorsal arter...,Venous anastomosis
7,Venous anastomosis of Left Gluteal artery perf...,Venous anastomosis
8,Venous anastomosis of Left Gluteal artery perf...,Venous anastomosis
9,Venous anastomosis of Left Serratus anterior p...,Venous anastomosis


,anastLabel,anastomosisTypeLabel
0,Venous anastomosis of Right Serratus anterior ...,Venous anastomosis
1,Venous anastomosis of Left Thoracodorsal arter...,Venous anastomosis
2,Venous anastomosis of Left Posterior tibial ar...,Venous anastomosis
3,Venous anastomosis of Right Lateral thoracic p...,Venous anastomosis
4,Venous anastomosis of Right Serratus anterior ...,Venous anastomosis
5,Venous anastomosis of Left Gluteal artery perf...,Venous anastomosis
6,Venous anastomosis of Left Thoracodorsal arter...,Venous anastomosis
7,Venous anastomosis of Left Gluteal artery perf...,Venous anastomosis
8,Venous anastomosis of Left Gluteal artery perf...,Venous anastomosis
9,Venous anastomosis of Left Serratus anterior p...,Venous anastomosis


### CQ17 — Flap Survival
*Did the flap fully survive?*

In [39]:
run("""
SELECT ?flapLabel ?outcomeLabel
WHERE {
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?outcomeLabel }
} LIMIT 10
""", title="CQ17 ABox — flap individuals by survival outcome")


CQ17 ABox — flap individuals by survival outcome
  10 row(s)


,flapLabel,outcomeLabel
0,Left Gluteus flap with gluteus maximus of pati...,Flaps without tissue loss
1,Right Scapular flap with skin of back of patie...,Flaps without tissue loss
2,Right Groin flap with abdominal fascia of pati...,Flaps without tissue loss
3,Right Gluteal artery perforator flap with subc...,Flaps without tissue loss
4,Right Vy flaps of the fingertip flap with subc...,Flaps without tissue loss
5,Left Lateral arm flap with brachialis of patie...,Flaps without tissue loss
6,Right Trapezius flap with skin of back of pati...,Flaps without tissue loss
7,Left Gluteus flap with skin of buttock of pati...,Flaps without tissue loss
8,Right Serratus anterior muscle flap with subcu...,Flaps without tissue loss
9,Left Lateral arm flap with subcutaneous adipos...,Flaps without tissue loss


,flapLabel,outcomeLabel
0,Left Gluteus flap with gluteus maximus of pati...,Flaps without tissue loss
1,Right Scapular flap with skin of back of patie...,Flaps without tissue loss
2,Right Groin flap with abdominal fascia of pati...,Flaps without tissue loss
3,Right Gluteal artery perforator flap with subc...,Flaps without tissue loss
4,Right Vy flaps of the fingertip flap with subc...,Flaps without tissue loss
5,Left Lateral arm flap with brachialis of patie...,Flaps without tissue loss
6,Right Trapezius flap with skin of back of pati...,Flaps without tissue loss
7,Left Gluteus flap with skin of buttock of pati...,Flaps without tissue loss
8,Right Serratus anterior muscle flap with subcu...,Flaps without tissue loss
9,Left Lateral arm flap with subcutaneous adipos...,Flaps without tissue loss


---
## Task-Based Evaluation

Five realistic clinical and research tasks demonstrating the ontology supports end-to-end use cases beyond abstract CQ answering.

### Task 1 — Flap selection by donor region
*Pedicled flap for trunk reconstruction — which flap types originate from the back of the trunk?*

In [40]:
_task1 = """
SELECT ?flapLabel ?survivalLabel
WHERE {
  VALUES ?procType { CLS }
  ?flap rdf:type owl:NamedIndividual ;
        ofl:OFLID12003 ?origin .
  ?origin rdfs:label ?originLabel .
  FILTER(CONTAINS(LCASE(?originLabel), "back of trunk"))
  ?flap obo:RO_0000056 ?process .
  ?process rdf:type ?procType .
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} LIMIT 10
"""
run(_task1.replace("CLS", pedicled_cls),
    title="Task 1 — Pedicled back-of-trunk flap individuals with survival outcome")


Task 1 — Pedicled back-of-trunk flap individuals with survival outcome
  10 row(s)


,flapLabel,survivalLabel
0,Left Latissimus dorsi flap with subcutaneous a...,Flaps with arterial obstruction
1,Right Latissimus dorsi flap with subcutaneous ...,Flaps with loss at the apex
2,Right Latissimus dorsi flap with subcutaneous ...,Flaps with partial loss
3,Left Latissimus dorsi flap with subcutaneous a...,Flaps with loss at the apex
4,Left Latissimus dorsi flap with subcutaneous a...,Flaps with partial loss
5,Left Latissimus dorsi flap with subcutaneous a...,Flaps with loss at the apex
6,Left Latissimus dorsi flap with subcutaneous a...,Flaps with partial loss
7,Left Latissimus dorsi flap with subcutaneous a...,Flaps with total loss
8,Left Latissimus dorsi flap with subcutaneous a...,Flaps with total loss
9,Left Latissimus dorsi flap with subcutaneous a...,Flaps with arterial obstruction


,flapLabel,survivalLabel
0,Left Latissimus dorsi flap with subcutaneous a...,Flaps with arterial obstruction
1,Right Latissimus dorsi flap with subcutaneous ...,Flaps with loss at the apex
2,Right Latissimus dorsi flap with subcutaneous ...,Flaps with partial loss
3,Left Latissimus dorsi flap with subcutaneous a...,Flaps with loss at the apex
4,Left Latissimus dorsi flap with subcutaneous a...,Flaps with partial loss
5,Left Latissimus dorsi flap with subcutaneous a...,Flaps with loss at the apex
6,Left Latissimus dorsi flap with subcutaneous a...,Flaps with partial loss
7,Left Latissimus dorsi flap with subcutaneous a...,Flaps with total loss
8,Left Latissimus dorsi flap with subcutaneous a...,Flaps with total loss
9,Left Latissimus dorsi flap with subcutaneous a...,Flaps with arterial obstruction


### Task 2 — Free flap survival audit
*Retrieve free flap individuals with their survival outcome.*

In [41]:
run("""
SELECT ?flapLabel ?survivalLabel
WHERE {
  ?flap rdf:type owl:NamedIndividual ;
        obo:RO_0000056 ?process .
  ?process rdf:type ofl:OFLID1000137 .
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} LIMIT 10
""", title="Task 2 — Free flap individuals with survival outcome")


Task 2 — Free flap individuals with survival outcome
  10 row(s)


,flapLabel,survivalLabel
0,Right Gluteal artery perforator flap with subc...,Flaps without tissue loss
1,Right Deep inferior epigastric perforator flap...,Flaps with venous obstruction
2,Left Serratus anterior perforator flap with su...,Flaps with venous obstruction
3,Left Thoracodorsal artery perforator flap with...,Flaps with loss at the apex
4,Left Thoracodorsal artery perforator flap with...,Flaps with partial loss
5,Left Deep inferior epigastric perforator flap ...,Flaps with arterial obstruction
6,Right Gluteal artery perforator flap with skin...,Flaps with arterial obstruction
7,Right Serratus anterior perforator flap with s...,Flaps with arterial obstruction
8,Left Gluteal artery perforator flap with skin ...,Flaps with total loss
9,Right Serratus anterior perforator flap with s...,Flaps with venous obstruction


,flapLabel,survivalLabel
0,Right Gluteal artery perforator flap with subc...,Flaps without tissue loss
1,Right Deep inferior epigastric perforator flap...,Flaps with venous obstruction
2,Left Serratus anterior perforator flap with su...,Flaps with venous obstruction
3,Left Thoracodorsal artery perforator flap with...,Flaps with loss at the apex
4,Left Thoracodorsal artery perforator flap with...,Flaps with partial loss
5,Left Deep inferior epigastric perforator flap ...,Flaps with arterial obstruction
6,Right Gluteal artery perforator flap with skin...,Flaps with arterial obstruction
7,Right Serratus anterior perforator flap with s...,Flaps with arterial obstruction
8,Left Gluteal artery perforator flap with skin ...,Flaps with total loss
9,Right Serratus anterior perforator flap with s...,Flaps with venous obstruction


### Task 3 — Mathes-Nahai Type I flap lookup
*Which muscle flap individuals have a single dominant vascular pedicle (Mathes-Nahai Type I)?*

In [42]:
_task3 = """
SELECT ?flapLabel ?mnTypeLabel
WHERE {
  VALUES (?mnType ?flapType) { PAIRS }
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ?flapType .
  OPTIONAL { ?mnType rdfs:label ?mnTypeLabel }
  OPTIONAL { ?flap   rdfs:label ?flapLabel }
} LIMIT 10
"""
run(_task3.replace("PAIRS", mn_task3_pairs),
    title="Task 3 — ABox flap individuals by Mathes-Nahai vascular pattern type")


Task 3 — ABox flap individuals by Mathes-Nahai vascular pattern type
  10 row(s)


,flapLabel,mnTypeLabel
0,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
1,Left Rectus abdominis flap with seventh rib of...,Type III: two dominant pedicles
2,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
3,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
4,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
5,Left Rectus abdominis flap with seventh rib of...,Type III: two dominant pedicles
6,Left Rectus abdominis flap with seventh rib of...,Type III: two dominant pedicles
7,Component sub-flap of Chimeric Rectus abdomini...,Type III: two dominant pedicles
8,Chimeric Rectus abdominis flap with seventh ri...,Type III: two dominant pedicles
9,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles


,flapLabel,mnTypeLabel
0,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
1,Left Rectus abdominis flap with seventh rib of...,Type III: two dominant pedicles
2,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
3,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
4,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles
5,Left Rectus abdominis flap with seventh rib of...,Type III: two dominant pedicles
6,Left Rectus abdominis flap with seventh rib of...,Type III: two dominant pedicles
7,Component sub-flap of Chimeric Rectus abdomini...,Type III: two dominant pedicles
8,Chimeric Rectus abdominis flap with seventh ri...,Type III: two dominant pedicles
9,Right Rectus abdominis flap with seventh rib o...,Type III: two dominant pedicles


### Task 4 — Prefabrication outcome tracking
*Pre-harvest modified flap individuals cross-tabulated with survival outcome.*

In [43]:
run("""
SELECT ?flapLabel ?survivalLabel
WHERE {
  ?flap rdf:type owl:NamedIndividual ;
        rdf:type ofl:OFLID10131 .
  VALUES ?survCls { ofl:OFLID10184 ofl:OFLID10183 ofl:OFLID10179
                   ofl:OFLID10178 ofl:OFLID10180 ofl:OFLID10181 }
  ?flap rdf:type ?survCls .
  OPTIONAL { ?flap    rdfs:label ?flapLabel }
  OPTIONAL { ?survCls rdfs:label ?survivalLabel }
} LIMIT 10
""", title="Task 4 — Survival outcomes for pre-harvest modified flaps")


Task 4 — Survival outcomes for pre-harvest modified flaps
  10 row(s)


,flapLabel,survivalLabel
0,Left Gracilis flap with fascia of thigh of pat...,Flaps with loss at the apex
1,Left Gracilis flap with fascia of thigh of pat...,Flaps with partial loss
2,Right Gracilis flap with skin of medial part o...,Flaps with loss at the apex
3,Right Gracilis flap with skin of medial part o...,Flaps with partial loss
4,Left Tensor fasciae latae flap with skin of la...,Flaps with total loss
5,Right Gracilis flap with fascia of thigh of pa...,Flaps with loss at the apex
6,Right Gracilis flap with fascia of thigh of pa...,Flaps with partial loss
7,Right Soleus flap with fibula of patient 1418,Flaps with total loss
8,Left Pectoralis major flap with sternum of pat...,Flaps with venous obstruction
9,Right Gluteal artery perforator flap with subc...,Flaps without tissue loss


,flapLabel,survivalLabel
0,Left Gracilis flap with fascia of thigh of pat...,Flaps with loss at the apex
1,Left Gracilis flap with fascia of thigh of pat...,Flaps with partial loss
2,Right Gracilis flap with skin of medial part o...,Flaps with loss at the apex
3,Right Gracilis flap with skin of medial part o...,Flaps with partial loss
4,Left Tensor fasciae latae flap with skin of la...,Flaps with total loss
5,Right Gracilis flap with fascia of thigh of pa...,Flaps with loss at the apex
6,Right Gracilis flap with fascia of thigh of pa...,Flaps with partial loss
7,Right Soleus flap with fibula of patient 1418,Flaps with total loss
8,Left Pectoralis major flap with sternum of pat...,Flaps with venous obstruction
9,Right Gluteal artery perforator flap with subc...,Flaps without tissue loss


### Task 5 — Operative planning: anatomical component inventory
*Component individuals of a generated Fibula flap instance with their FMA types.*

In [44]:
run("""
SELECT ?componentLabel ?typeLabel
WHERE {
  ?flap rdfs:label ?fl . FILTER(CONTAINS(LCASE(?fl), "fibula flap"))
  ?flap ofl:OFLID13296 ?component .
  OPTIONAL { ?component rdfs:label ?componentLabel }
  OPTIONAL {
    ?component rdf:type ?t .
    FILTER(STRSTARTS(STR(?t), "http://purl.obolibrary.org/obo/FMA_"))
    OPTIONAL { ?t rdfs:label ?typeLabel }
  }
} LIMIT 10
""", title="Task 5 — Component individuals of a generated Fibula flap ABox instance")


Task 5 — Component individuals of a generated Fibula flap ABox instance
  10 row(s)


,componentLabel,typeLabel
0,Flap harvest process of Right Fibula flap with...,—
1,Pedicled flap transfer process of Right Fibula...,—
2,Preinsertion treatment of recipient site for R...,—
3,Flap harvest process of Right Fibula flap with...,—
4,Pedicled flap transfer process of Right Fibula...,—
5,Preinsertion treatment of recipient site for R...,—
6,Flap harvest process of Left Fibula flap with ...,—
7,Pedicled flap transfer process of Left Fibula ...,—
8,Preinsertion treatment of recipient site for L...,—
9,Flap harvest process of Left Fibula flap with ...,—


,componentLabel,typeLabel
0,Flap harvest process of Right Fibula flap with...,—
1,Pedicled flap transfer process of Right Fibula...,—
2,Preinsertion treatment of recipient site for R...,—
3,Flap harvest process of Right Fibula flap with...,—
4,Pedicled flap transfer process of Right Fibula...,—
5,Preinsertion treatment of recipient site for R...,—
6,Flap harvest process of Left Fibula flap with ...,—
7,Pedicled flap transfer process of Left Fibula ...,—
8,Preinsertion treatment of recipient site for L...,—
9,Flap harvest process of Left Fibula flap with ...,—
